## Mini Project: Dry Bean Type Classification

An agriculture company approaches you with a challenge: they are currently classifying different types of dry beans manually, a process that is labour-intensive, prone to errors and inefficient at scale. They want to automate this classification process using Artificial Intelligence to improve accuracy, reduce operational costs and ensure consistent quality in packaging and distribution.

As a data scientist, your role is to help them build a machine learning solution that can accurately classify bean types based on physical characteristics such as area, perimeter, shape and compactness. By doing this, you'll:
-	Automate the bean classification process using supervised learning techniques
-	Help the company reduce manual labour and cost
-	Improve the speed and reliability of quality control operations
-	Deliver a scalable solution for real-time classification in industrial settings
This project not only enhances your understanding of supervised machine learning but also demonstrates how AI can be directly applied to Agri-tech and food processing industries for tangible business impact.

Data Dictionary
All configurations (like area, shape and roundness) of data were gathered through camera-based systems using computer vision algorithms—a method commonly used in modern agricultural quality control setups.
Attribute Information:
1.	Area (A): The area of a bean zone and the number of pixels within its boundaries.
2.	Perimeter (P): Bean circumference is defined as the length of its border.
3.	Major axis length (L): The distance between the ends of the longest line that can be drawn from a bean.
4.	Minor axis length (l): The longest line that can be drawn from the bean while standing perpendicular to the main axis.
5.	Aspect ratio (K): Defines the relationship between L and l.
6.	Eccentricity (Ec): Eccentricity of the ellipse having the same moments as the region.
7.	Convex area (C): Number of pixels in the smallest convex polygon that can contain the area of a bean seed.
8.	Equivalent diameter (Ed): The diameter of a circle having the same area as a bean seed area.
9.	Extent (Ex): The ratio of the pixels in the bounding box to the bean area.
10.	Solidity (S): Also known as convexity. The ratio of the pixels in the convex shell to those found in beans.
11.	Roundness (R): Calculated with the following formula: (4piA)/(P^2)
12.	Compactness (CO): Measures the roundness of an object: Ed/L
13.	ShapeFactor1 (SF1)
14.	ShapeFactor2 (SF2)
15.	ShapeFactor3 (SF3)
16.	ShapeFactor4 (SF4)
17.	Class (Seker, Barbunya, Bombay, Cali, Dermosan, Horoz and Sira) 

1. Import and Load the Data
-	Import necessary libraries (pandas, numpy, matplotlib, seaborn, sklearn etc.)

In [ ]:
import pickle
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import PowerTransformer, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import  accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

-	Load the dataset and explore it using .head(), .info(), and .describe()

In [ ]:
DF_DB = pd.read_excel('Dry_Bean_Dataset.xlsx')
display(DF_DB.head())
display(DF_DB.info())
display(DF_DB.describe())

2. Exploratory Data Analysis (EDA)
-	Visualize distributions of features using histograms and boxplots
-	Analyze the class distribution (check for class imbalance)
-	Plot feature correlations (eg heatmap)
-	Visualize multivariate relationships (pairplot)
-	Summarize key findings

### Data Distribution of Each Feature (Boxplot)

In [ ]:
num_cols = DF_DB.select_dtypes(include=[np.number]).columns.tolist()

plt.figure(figsize=(15, 4 * len(num_cols)))

for i, col in enumerate(num_cols, 1):
    plt.subplot(len(num_cols), 1, i)
    
    sns.boxplot(x=DF_DB[col], color='skyblue')
    
    mean_val = DF_DB[col].mean()
    median_val = DF_DB[col].median()
    plt.axvline(mean_val, color='green', linestyle='--', label=f'Mean: {mean_val:.2f}')
    plt.axvline(median_val, color='orange', linestyle='-', label=f'Median: {median_val:.2f}')
    
    plt.title(f'Box Plot Distribution & Outliers: {col}', fontsize=12, fontweight='bold')
    plt.legend()

plt.tight_layout()
plt.show()

1. Area (Top): Heavily right-skewed with a compact interquartile range (IQR) near the lower end and a long tail of extreme high-value outliers.

2. Perimeter: Right-skewed distribution featuring a moderate IQR and a dense trail of extended outliers on the upper side.

3. Major Axis Length: Right-skewed with a wider central spread and a prominent upper tail of outliers.

4. Minor Axis Length: Right-skewed with a narrow box shifted towards lower values and a stretched upper whisker with outliers.

5. Aspect Ratio: Moderately right-skewed, with its box in the lower-middle range and a concentrated cluster of high-value outliers.

6. Eccentricity: Left-skewed, where the box is positioned toward the higher end with a long tail of lower-value outliers extending to the left.

7. Convex Area: Strongly right-skewed, showing a tight IQR near the lower end and an extensive trail of high-end outliers.

8. Equivalent Diameter: Right-skewed with a compact box on the lower-left and a significant spread of upper outliers.

9. Extent: Slightly left-skewed, featuring a wide box positioned toward the right and minor low-value outliers.

10. Solidity:Strongly left-skewed, with a very narrow IQR pressed tightly against the maximum value and a long trail of low-value outliers.

11. Roundness: Left-skewed, with the box situated on the right side and a trail of outliers stretching past the left whisker.

12. Compactness: Approximately symmetric, with the mean and median closely overlapping at the center and minor outliers on both extremes.

13. Shape Factor 1: Nearly symmetric, showing a balanced box and whiskers with minimal outliers on both ends.

14. Shape Factor 2: Symmetric and uniformly spread, featuring a wide box, balanced whiskers, overlapping central tendencies, and **no visible outliers**.

15. Shape Factor 3: Slightly right-skewed, with the box in the lower-middle region and a small detached cluster of outliers on the far right.

16. Shape Factor 4: Extremely left-skewed, with a tight IQR pressed against the far right edge and an extensive trail of low-value outliers.

### Skewness of Data Features (Histogram)

In [ ]:
DF_DB_NUM = DF_DB.select_dtypes(include=[np.number])

skewness = DF_DB_NUM.skew().sort_values(ascending=False)
display(DF_DB_NUM.skew().sort_values(ascending=False))

cols = DF_DB_NUM.skew().index

for col in cols:
    sns.histplot(data=DF_DB_NUM, x=col, kde=True, color="#8D36F7")
    plt.title(f'{col} (Skewness: {skewness[col]:.2f})')
    plt.show()

Area:

* Distribution Shape: The feature exhibits a strong right-skewness with a skewness coefficient of 2.95.
* Central Tendency & Frequency: The highest density (count exceeding 800) is heavily concentrated between approximately 30,000 and 50,000.
* Secondary Bump: There is a smaller secondary peak or bump visible around 65,000 to 75,000 before the distribution tails off.
* Long Tail / Outliers: A sparse, extended tail stretches far to the right up to around 250,000, which corresponds to the outliers observed previously in the box plot.

Perimeter:

* Distribution Shape: The feature exhibits a right-skewness coefficient of 1.63.
* Primary Concentration: The highest frequency of data points is concentrated between roughly 650 and 800, with a peak count exceeding 900.
* Secondary Mode: A distinct secondary bump is visible in the distribution around 1,000 to 1,100.
* Right Tail: A sparse, extended tail stretches toward the higher values, reaching up to approximately 2,000.

MajorAxisLength:

* Distribution Shape: The MajorAxisLength feature exhibits a moderate positive (right) skewness with a skewness value of 1.36.
* Primary Concentration: The highest frequency of observations is tightly clustered between 220 and 280, reaching a peak count of over 1,100 around 250.
* Secondary Mode: A prominent secondary peak is present between 360 and 420 (peaking near 390 with counts close to 575), indicating a clear bimodal pattern that likely distinguishes specific sub-categories within your 6 classes.
* Right Tail: Beyond 460, the distribution tapers off into a sparse tail extending up to roughly 740, with a minor cluster of high-value outliers around 550 to 650.

MinorAxisLength:

* Distribution Shape: The MinorAxisLength feature exhibits a strong positive (right) skewness with a skewness coefficient of 2.24.
* Primary Concentration: The highest frequency of observations is tightly clustered between 160 and 210, peaking just above 830 around 190–195.
* Secondary Mode: A distinct secondary bump appears around 225 to 245 (peaking just under 300), suggesting a subgroup separation among your categories.
* Detached Outlier Cluster: After tapering near zero around 300, a noticeable detached cluster of high-value observations emerges between 340 and 420 (peaking around 370), extending up to roughly 460.

AspectRation:

* Distribution Shape: The AspectRation feature shows a mild positive (right) skewness with a coefficient of 0.58.
* Multi-Modal Structure: The distribution displays three distinct peaks (trimodal pattern) rather than a single bell curve, which strongly indicates natural separation among your categories:
* Primary Peak: The largest concentration sits between 1.45 and 1.65, reaching a peak count near 700 around 1.50–1.55.
* Lower Secondary Peak: A prominent shoulder appears on the left around 1.20 to 1.28, peaking close to 280.
* Upper Secondary Peak: A broader, flatter bump appears on the right between 1.95 and 2.15, with counts hovering around 130–140.
* Overall Spread: Values span smoothly from roughly 1.02 up to 2.43 without extreme, detached outliers.

Eccentricity:

* Distribution Shape: The Eccentricity feature exhibits a moderate negative (left) skewness with a skewness coefficient of -1.06.
* Primary Concentration: The highest frequency of data points is concentrated between 0.72 and 0.82, reaching a peak count of roughly 670 around 0.76–0.77.
* Secondary Peaks: A sharp secondary peak appears near the upper end around 0.87–0.88 (with counts reaching approximately 370), alongside a flatter shoulder on the left between 0.55 and 0.65 (hovering around counts of 100–115).
* Left Tail: The distribution tapers gradually toward the lower values, forming a long tail extending down to roughly 0.22.

ConvexArea:

* Distribution Shape: The ConvexArea feature exhibits a strong positive (right) skewness with a skewness coefficient of 2.94.
* Primary Concentration: The highest density of observations is clustered between approximately 30,000 and 50,000, peaking above 900 around 38,000–40,000.
* Secondary Bump: A smaller secondary bump appears between 65,000 and 80,000 (peaking just under 300) before tapering down toward 105,000.
* Detached Right Tail: A sparse, extended tail stretches from around 130,000 up to roughly 260,000 (peaking slightly around 170,000), closely mirroring the distribution of Area and indicating strong multicollinearity between the two features.

EquivDiameter:

* Distribution Shape: The EquivDiameter feature exhibits a strong positive (right) skewness with a skewness coefficient of 1.95.
* Primary Concentration: The highest frequency of observations is tightly clustered between 200 and 250, peaking at over 830 around 225.
* Secondary Mode: A clear secondary bump appears between 290 and 315 (peaking just under 350), reflecting sub-category separation within the dataset.
* Detached Right Tail: After tapering down near zero around 375, a small, detached cluster of high-value observations emerges between 410 and 530 (peaking slightly around 465–475), extending up to roughly 570.

Extent:

* Distribution Shape: The Extent feature exhibits a moderate negative (left) skewness with a skewness coefficient of -0.90.
* Primary Concentration: The highest density of observations is concentrated toward the upper end between 0.76 and 0.80, reaching a peak count of nearly 800 around 0.78.
* Secondary Shoulder: A distinct secondary bump or shoulder appears in the middle range between 0.70 and 0.73, peaking around 450.
* Left Tail: The distribution tapers down toward the lower values, forming a gradual tail extending from roughly 0.66 down to 0.56 with low counts (under 70).

Solidity:

* Distribution Shape: The Solidity feature exhibits a strong negative (left) skewness with a skewness coefficient of -2.55.
* Primary Concentration: The vast majority of observations are tightly concentrated near the upper boundary between 0.985 and 0.993, peaking sharply above 750 around 0.989–0.990.
* Unimodal Peak: Unlike several previous features, Solidity shows a single, sharp peak rather than distinct secondary bumps.
* Left Tail: Below 0.980, counts drop rapidly into a very thin, extended left tail reaching all the way down to roughly 0.92, representing sparse low-value outliers.

Roundness:

* Distribution Shape: The roundness feature exhibits a moderate negative (left) skewness with a skewness coefficient of -0.64.
* Primary Concentration: The highest density of observations is concentrated between 0.87 and 0.93, reaching a peak count of roughly 740 around 0.90–0.91.
* Secondary Shoulder: A distinct secondary bump appears on the left slope between 0.78 and 0.83, peaking just below 400 around 0.81.
* Left Tail: Below 0.75, frequencies drop steadily into a thin left tail extending down to roughly 0.49.

Compactness:

* Distribution Shape: The Compactness feature is nearly symmetric with a skewness coefficient of 0.04.
* Trimodal Structure: While balanced overall, the distribution exhibits three distinct peaks rather than a single normal curve, reflecting clear separation among your categories:
* Central Primary Peak: The highest concentration sits between 0.78 and 0.83, peaking at approximately 700 around 0.80–0.81.
* Left Secondary Peak: A smaller, distinct bump appears around 0.69 to 0.71, reaching a count of roughly 220.
* Right Secondary Peak: A symmetrical counterpart bump appears around 0.89 to 0.91, peaking just above 210.
* Overall Spread: Values span evenly from roughly 0.64 to 0.99 without extreme, detached outliers.

ShapeFactor1:

* Distribution Shape: The ShapeFactor1 feature exhibits a mild negative (left) skewness with a skewness coefficient of -0.53.
* Trimodal Structure: The distribution displays three distinct peaks, reflecting clear separation among different categories in the dataset:
* Primary Peak: The largest concentration lies between 0.0063 and 0.0070, reaching a peak count just above 800 around 0.0066–0.0067.
* Middle Secondary Peak: A prominent shoulder appears between 0.0052 and 0.0058, peaking near 380 around 0.0055.
* Detached Left Cluster: A separate, smaller peak emerges at the lower end between 0.0030 and 0.0039 (peaking just over 100 around 0.0035), isolated from the main body by a gap near 0.0041.
* Overall Spread: Values range from roughly 0.0028 up to 0.0105, tapering smoothly on the right side beyond 0.0090.

ShapeFactor2:

* Distribution Shape: The ShapeFactor2 feature exhibits a mild positive (right) skewness with a skewness coefficient of 0.30.
* Bimodal Structure: The distribution displays two distinct peaks, highlighting strong separation among different categories in your dataset:
* Primary Peak: A sharp, narrow peak is concentrated at the lower end between 0.0009 and 0.0012, reaching a count of over 1,000 around 0.0010–0.0011.
* Broad Secondary Peak: A much wider, flatter mound spans from roughly 0.0015 to 0.0024, peaking around 635 near 0.0017–0.0018.
* Overall Spread: Values range smoothly from roughly 0.00055 up to 0.00365, tapering gradually on the right without detached outliers.

ShapeFactor3:

* Distribution Shape: The ShapeFactor3 feature is nearly symmetric with a slight positive (right) skewness coefficient of 0.24.
* Trimodal Structure: Similar to Compactness, the distribution displays three distinct peaks, highlighting clear subgroup separation across categories:
* Central Primary Peak: The main concentration sits between 0.61 and 0.68, reaching a peak count just under 700 around 0.65.
* Left Secondary Peak: A distinct shoulder emerges on the lower end around 0.47 to 0.50, peaking near 255.
* Right Secondary Peak: A broader secondary bump appears toward the upper end between 0.78 and 0.84, peaking just below 200 around 0.81.
* Overall Spread: Values span continuously from roughly 0.41 up to 0.98 without detached outliers.

ShapeFactor4:

* Distribution Shape: The ShapeFactor4 feature exhibits a strong negative (left) skewness with a skewness coefficient of -2.76.
* Primary Concentration: The vast majority of observations are tightly packed near the upper boundary between 0.995 and 0.999, reaching a sharp peak count close to 900 around 0.998.
* Unimodal Peak: Like Solidity, this feature displays a single, sharp peak right against the upper limit rather than distinct secondary bumps.
* Left Tail: Below 0.990, counts drop steeply into a long, sparse left tail extending down to roughly 0.948, representing low-value outliers.

### Class Distribution of Data Features (Imbalance Check)

In [ ]:
DF_DB['Class'] = DF_DB['Class'].astype('category')
display(DF_DB['Class'].value_counts())
plt.figure(figsize=(8, 6))
BarChart = plt.bar(DF_DB['Class'].value_counts().index.to_numpy(), DF_DB['Class'].value_counts(), color='skyblue')
plt.title('Class Distribution', fontsize=14, fontweight='bold')
plt.xlabel('Class')
plt.ylabel('Count')
plt.bar_label(BarChart, labels=DF_DB['Class'].value_counts().to_numpy(), padding=2, fontsize=9)
plt.show()

7-Class Structure: The target variable actually consists of 7 distinct bean varieties (DERMASON, SIRA, SEKER, HOROZ, CALI, BARBUNYA, and BOMBAY) totaling 13,611 samples.

Significant Class Imbalance: The dataset is noticeably imbalanced, with the majority class (DERMASON at 3,546 instances) having nearly 6.8 times more samples than the smallest minority class (BOMBAY at 522 instances).

Distribution Tiering:
* High Representation: DERMASON (3,546) and SIRA (2,636) together account for over 45% of the entire dataset.
* Moderate Representation: SEKER (2,027), HOROZ (1,928), CALI (1,630), and BARBUNYA (1,322) form a steady middle tier.
* Low Representation: BOMBAY (522) is heavily underrepresented, which directly explains the small detached outlier clusters seen earlier in size features like Area, Perimeter, and MinorAxisLength.

### Correlation Check (Heatmap)

In [ ]:
Corr_Matrix = DF_DB.select_dtypes(include=[np.number]).corr()

display(Corr_Matrix)

plt.figure(figsize=(12, 8))

sns.heatmap(Corr_Matrix,annot=True,cmap='coolwarm',center=0,fmt='.2f',linewidths=0.5)

plt.title('Correlation Heatmap')
plt.tight_layout()
plt.show()

* Strong Multicollinearity Clusters: Area, Perimeter, MajorAxisLength, MinorAxisLength, ConvexArea, and EquivDiameter exhibit extremely high positive correlations with each other (values generally ranging from 0.83 to 1.00), forming a dense cluster of redundant geometric measures. Similarly, AspectRation and Eccentricity show an intense positive correlation of 0.92, while Compactness and ShapeFactor3 correlate at 1.00.

* Strong Inverse Relationships: AspectRation and Eccentricity both show near-perfect negative correlations with Compactness (-0.99 and -0.97 respectively) and ShapeFactor3 (-0.98 for both), indicating that as elongation increases, compactness and shape factor metrics drop sharply. MinorAxisLength also displays a strong negative correlation of -0.95 with ShapeFactor1.

* Independent or Low-Correlation Features: Extent stands out with generally weak correlation coefficients across most other variables (mostly hovering between -0.37 and 0.35), suggesting it captures distinct variance that is not redundantly explained by the primary size and shape measures.

### Visualize multivariate relationships (pairplot)

In [ ]:
sns.pairplot(DF_DB, hue='Class', palette='Set2', diag_kind='kde')
plt.show()

3. Missing Values & Outlier Treatment
-	Check for and handle missing values
-	Detect and treat outliers if needed (Z-score / IQR methods/boxplots)

### Missing Values and Outliers

In [ ]:
display(DF_DB.isnull().sum())
display(DF_DB.select_dtypes(include=['str']).apply(lambda x: x.unique()))

for col in DF_DB.select_dtypes(include=[np.number]).columns:
    Q1 = DF_DB[col].quantile(0.25)
    Q3 = DF_DB[col].quantile(0.75)
    IQR_Range = Q3 - Q1
    display('Outliers for Column ' + col, DF_DB.loc[((DF_DB[col] < (Q1 - 1.5 * IQR_Range)) | (DF_DB[col] > (Q3 + 1.5 * IQR_Range)))])
    display('Extreme Outliers for Column ' + col , DF_DB.loc[((DF_DB[col] < (Q1 - 1.72 * IQR_Range)) | (DF_DB[col] > (Q3 + 1.72 * IQR_Range)))])
    print()

plt.figure(figsize=(12, 6))
sns.heatmap(DF_DB.isnull(), cbar=False, cmap='viridis')
plt.title('Null Values Heatmap')
plt.show()

4. Feature Engineering & Preprocessing
-	Scale numerical features (StandardScaler / MinMaxScaler)
-	Encode categorical variables if necessary
-	Check and treat skewness if required
-	Split data into train/test sets (use stratified sampling while splitting)


### Feature Scaling (Standard Scaler), Encoding Y-Categorical and Splitting Data into Test and Train with Stratified Sampling

In [ ]:
DF_DB_X = DF_DB.drop(columns=['Class'])
y_encoded = DF_DB['Class'].astype('category').cat.codes
X_train, X_test, y_train, y_test = train_test_split(
    DF_DB_X, y_encoded, test_size=0.15, random_state=42, stratify=y_encoded
)
skewness = DF_DB_X.skew().sort_values(ascending=False)
display(skewness)
cols = DF_DB_X.skew()[(DF_DB_X.skew() > 0.5) | (DF_DB_X.skew() < -0.5)].index
remaining_cols = [col for col in DF_DB_X.columns if col not in cols]
CT = ColumnTransformer([
    ("PT", PowerTransformer(method='yeo-johnson'), cols),
    ("ss", StandardScaler(), remaining_cols)
])
X_train_Transformed = CT.fit_transform(X_train)
X_test_Transformed = CT.transform(X_test)

5. Model Building: Try Multiple Classifiers, Train and test the dataset on a variety of supervised classification algorithms:
-	Logistic Regression
-	Decision Tree Classifier
-	Random Forest Classifier
-	K-Nearest Neighbors (KNN)
-	Support Vector Machine (SVM)
-	Ensemble Learning Methods
-	Naive Bayes and more…

Use Cross validation techniques to check if model performance is improved.


### Training and Testing Multiple Models on Base Parameters Prefered for the Dataset

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(random_state=42),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=10, min_samples_leaf=2, min_samples_split=2, random_state=42, class_weight='balanced'
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=100, max_depth=10, min_samples_leaf=2, min_samples_split=5,
        max_features=0.8, random_state=42, n_jobs=-1, class_weight='balanced'
    ),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
    "SVC": SVC(),
    "KNN": KNeighborsClassifier(n_neighbors=7),
    "XG" : XGBClassifier(n_estimators=100, max_depth=10, learning_rate=0.1, random_state=42, n_jobs=-1, subsample=0.8, colsample_bytree=0.8)
}

results = []
fitted_models = {}

category_mapping = dict(enumerate(DF_DB['Class'].astype('category').cat.categories))
print(category_mapping)

for name, estimator in models.items():
    estimator.fit(X_train_Transformed,y_train)

    cv_acc = cross_val_score(
        estimator, X_train_Transformed, y_train,
        cv=3, scoring="accuracy", n_jobs=-1
    ).mean()

    train_pred = estimator.predict(X_train_Transformed)
    test_pred = estimator.predict(X_test_Transformed)
    display('Confusion Matrix for ' + name, confusion_matrix(y_test, test_pred))
    display('Classification Report for ' + name)
    print(classification_report(y_test, test_pred))

    cm = confusion_matrix(y_test, test_pred)

    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)

    plt.title(name + ' Confusion Matrix Heatmap')
    plt.xlabel('Predicted Class')
    plt.ylabel('Actual Class')
    plt.show()

    results.append({
        "Model": name,
        "CV Accuracy": cv_acc,
        "Train Accuracy": accuracy_score(y_train, train_pred),
        "Test Accuracy": accuracy_score(y_test, test_pred),
        "Precision": precision_score(y_test, test_pred, average='weighted'),
        "Recall": recall_score(y_test, test_pred, average='weighted'),
        "F1 Score": f1_score(y_test, test_pred, average='weighted'),
        "Overfitting (Y/N)": "Yes" if accuracy_score(y_train, train_pred) > 0.98 and (accuracy_score(y_train, train_pred) - accuracy_score(y_test, test_pred)) > 0.05 else "No"
    })

results_df = pd.DataFrame(results).sort_values(["F1 Score", "CV Accuracy"], ascending=False).reset_index(drop=True)
display(results_df.round(3))

6. Handling Class Imbalance, Apply techniques like:
 -	SMOTE (Synthetic Minority Over-sampling)
 -	Random Oversampling / Undersampling
 -	Class weighting

Evaluate if performance improves on minority classes

In [ ]:
smote = SMOTE(random_state=42)
resampled_data = smote.fit_resample(X_train_Transformed, y_train)

X_train_smote, y_train_smote = resampled_data[0], resampled_data[1]

DF_X_train_smote = pd.DataFrame(X_train_smote, columns=DF_DB_X.columns)
DF_Y_train_smote = pd.Series(list(y_train_smote), name='Class')


models = {
    "Logistic Regression": LogisticRegression(random_state=42),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=10, min_samples_leaf=2, min_samples_split=2, random_state=42, class_weight='balanced'
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=100, max_depth=10, min_samples_leaf=2, min_samples_split=5,
        max_features=0.8, random_state=42, n_jobs=-1, class_weight='balanced'
    ),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
    "SVC": SVC(),
    "KNN": KNeighborsClassifier(n_neighbors=7),
    "XG" : XGBClassifier(n_estimators=100, max_depth=10, learning_rate=0.1, random_state=42, n_jobs=-1, subsample=0.8, colsample_bytree=0.8)
}

results = []
fitted_models = {}

category_mapping = dict(enumerate(DF_DB['Class'].astype('category').cat.categories))
print(category_mapping)

for name, estimator in models.items():
    estimator.fit(DF_X_train_smote, DF_Y_train_smote)

    cv_acc = cross_val_score(
        estimator, DF_X_train_smote, DF_Y_train_smote,
        cv=3, scoring="accuracy", n_jobs=-1
    ).mean()

    train_pred = estimator.predict(DF_X_train_smote)
    test_pred = estimator.predict(X_test_Transformed)
    display('Confusion Matrix for ' + name, confusion_matrix(y_test, test_pred))
    display('Classification Report for ' + name)
    print(classification_report(y_test, test_pred))

    cm = confusion_matrix(y_test, test_pred)

    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)

    plt.title(name + ' Confusion Matrix Heatmap')
    plt.xlabel('Predicted Class')
    plt.ylabel('Actual Class')
    plt.show()

    results.append({
        "Model": name,
        "CV Accuracy": cv_acc,
        "Train Accuracy": accuracy_score(DF_Y_train_smote, train_pred),
        "Test Accuracy": accuracy_score(y_test, test_pred),
        "Precision": precision_score(y_test, test_pred, average='weighted'),
        "Recall": recall_score(y_test, test_pred, average='weighted'),
        "F1 Score": f1_score(y_test, test_pred, average='weighted'),
        "Overfitting (Y/N)": "Yes" if accuracy_score(DF_Y_train_smote, train_pred) > 0.98 and (accuracy_score(DF_Y_train_smote, train_pred) - accuracy_score(y_test, test_pred)) > 0.05 else "No"
    })

results_df = pd.DataFrame(results).sort_values(["F1 Score", "CV Accuracy"], ascending=False).reset_index(drop=True)
display(results_df.round(3))

### Hyperparameter Tuning on Default Data

### Hyperparameter tuning for Gradient XG Boost Classifier using GridSearchCV

In [ ]:
param_grid = {
    'n_estimators': [100, 125],
    'learning_rate': [0.05, 0.1, 0.2],
    'max_depth': [6, 7, 8, 9, 10],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

xgb_base = XGBClassifier(random_state=42)

grid_search = GridSearchCV(
    estimator=xgb_base,
    param_grid=param_grid,
    scoring='f1_macro',
    cv=5,
    n_jobs=-1,
    verbose=1
)
grid_search.fit(X_train_Transformed, y_train)

best_rf_grid = grid_search.best_estimator_
print("\n--- Best Hyperparameters Found ---")
print(grid_search.best_params_)

train_pred = best_rf_grid.predict(X_train_Transformed)
test_pred = best_rf_grid.predict(X_test_Transformed)

print("\n--- Tuned XGB Classifier Model Performance ---")
print(f"Train Accuracy: {accuracy_score(y_train, train_pred):.4f}")
print(f"Test Accuracy:  {accuracy_score(y_test, test_pred):.4f}")
print(f"Precision:      {precision_score(y_test, test_pred, average='weighted'):.4f}")
print(f"Recall:         {recall_score(y_test, test_pred, average='weighted'):.4f}")
print(f"F1 Score:       {f1_score(y_test, test_pred, average='weighted'):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, test_pred))
print(f"Overfitting:    {'Yes' if accuracy_score(y_train, train_pred) > 0.98 and (accuracy_score(y_train, train_pred) - accuracy_score(y_test, test_pred)) > 0.05 else 'No'}")

### Hyperparameter tuning for Random Forest Classifier using GridSearchCV

In [ ]:
param_grid = {
    'n_estimators': [100, 125],
    'max_depth': [6, 7, 8, 9, 10],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
    'max_features': ['sqrt', 1.0]
}

rf_base = RandomForestClassifier(random_state=42)

grid_search = GridSearchCV(
    estimator=rf_base,
    param_grid=param_grid,
    scoring='f1_macro',
    cv=5,
    n_jobs=-1,
    verbose=1
)
grid_search.fit(X_train_Transformed, y_train)

best_rf_grid = grid_search.best_estimator_
print("\n--- Best Hyperparameters Found ---")
print(grid_search.best_params_)

train_pred = best_rf_grid.predict(X_train_Transformed)
test_pred = best_rf_grid.predict(X_test_Transformed)

print("\n--- Tuned Random Forest Model Performance ---")
print(f"Train Accuracy: {accuracy_score(y_train, train_pred):.4f}")
print(f"Test Accuracy:  {accuracy_score(y_test, test_pred):.4f}")
print(f"Precision:      {precision_score(y_test, test_pred, average='weighted'):.4f}")
print(f"Recall:         {recall_score(y_test, test_pred, average='weighted'):.4f}")
print(f"F1 Score:       {f1_score(y_test, test_pred, average='weighted'):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, test_pred))
print(f"Overfitting:    {'Yes' if accuracy_score(y_train, train_pred) > 0.98 and (accuracy_score(y_train, train_pred) - accuracy_score(y_test, test_pred)) > 0.05 else 'No'}")

### Hyperparameter tuning for SVC using GridSearchCV

In [ ]:
param_grid = [
    { 'kernel': ['linear'], 'C': [0.1, 1, 10, 100], 'decision_function_shape': ['ovr']},
    { 'kernel': ['rbf'], 'C': [0.1, 1, 10, 100], 'gamma': ['scale', 'auto', 0.001, 0.01, 0.1, 1], 'decision_function_shape': ['ovr', 'ovo']},
    { 'kernel': ['poly'], 'C': [0.1, 1, 10], 'degree': [2, 3, 4], 'gamma': ['scale', 'auto', 0.1], 'decision_function_shape': ['ovr', 'ovo'] }
]

svc = SVC(random_state=42)

grid_search = GridSearchCV(
    estimator=svc,
    param_grid=param_grid,
    cv=5,
    scoring='f1_weighted',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train_Transformed, y_train)

best_svc_grid = grid_search.best_estimator_
print("\n--- Best Hyperparameters Found ---")
print(grid_search.best_params_)

train_pred = best_svc_grid.predict(X_train_Transformed)
test_pred = best_svc_grid.predict(X_test_Transformed)

print("\n--- Tuned SVC Model Performance ---")
print(f"Train Accuracy: {accuracy_score(y_train, train_pred):.4f}")
print(f"Test Accuracy:  {accuracy_score(y_test, test_pred):.4f}")
print(f"Precision:      {precision_score(y_test, test_pred, average='weighted'):.4f}")
print(f"Recall:         {recall_score(y_test, test_pred, average='weighted'):.4f}")
print(f"F1 Score:       {f1_score(y_test, test_pred, average='weighted'):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, test_pred))
print(f"Overfitting:    {'Yes' if accuracy_score(y_train, train_pred) > 0.98 and (accuracy_score(y_train, train_pred) - accuracy_score(y_test, test_pred)) > 0.05 else 'No'}")

### Hyperparameter tuning for Logistic Regression using GridSearchCV

In [ ]:
param_grid_lr = [
    {
        'solver': ['lbfgs', 'saga'],
        'C': [0.01, 0.1, 1, 10, 100],
        'max_iter': [4000]
    },
    {
        'solver': ['saga'],
        'l1_ratio': [0.0, 0.5, 1.0],
        'C': [0.01, 0.1, 1, 10, 100],
        'max_iter': [4000]
    }
]

lr = LogisticRegression(random_state=42)

grid_search_lr = GridSearchCV(
    estimator=lr,
    param_grid=param_grid_lr,
    cv=5,
    scoring='f1_weighted',
    n_jobs=-1,
    verbose=1
)

grid_search_lr.fit(X_train_Transformed, y_train)

best_lr_grid = grid_search_lr.best_estimator_
print("\n--- Best Hyperparameters Found (Logistic Regression) ---")
print(grid_search_lr.best_params_)

train_pred = best_lr_grid.predict(X_train_Transformed)
test_pred = best_lr_grid.predict(X_test_Transformed)

print("\n--- Tuned Logistic Regression Model Performance ---")
print(f"Train Accuracy: {accuracy_score(y_train, train_pred):.4f}")
print(f"Test Accuracy:  {accuracy_score(y_test, test_pred):.4f}")
print(f"Precision:      {precision_score(y_test, test_pred, average='weighted'):.4f}")
print(f"Recall:         {recall_score(y_test, test_pred, average='weighted'):.4f}")
print(f"F1 Score:       {f1_score(y_test, test_pred, average='weighted'):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, test_pred))
print(f"Overfitting:    {'Yes' if accuracy_score(y_train, train_pred) > 0.98 and (accuracy_score(y_train, train_pred) - accuracy_score(y_test, test_pred)) > 0.05 else 'No'}")

#### Here after Hyperparameter Tunning Logistic Regression is Best Model for Predicting in Training as well as Testing without Overfitting issue

### Exporting Model (SVC)

In [ ]:

model_dict = {
    "CT": CT,
    "model": best_lr_grid
}
with open("Dry_Beans_Model.pkl", "wb") as file:
    pickle.dump(model_dict, file)